# Minimal example
Run the size mitigation on a handful of spws in one call and show the final mitigation table.

Requires `smart_size_mitigation_heuristics.py` in the same directory (or on PYTHONPATH).

In [1]:
from smart_size_mitigation_heuristics import build_mitigation_table, Limits, plan

# 1. Define the input parameters (example case)

In [2]:
spws = [
    {  # representative spw: nbin-eligible, single pointing, robust=2.0
        "spw": "25", "nchan": 3840, "ratio_satisfied": False,
        "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
        "n_roi": 3000, "n_total": 3840,
    },
    {  # smaller nchan, only eligible for nbin if ratio condition met
        "spw": "27", "nchan": 960, "ratio_satisfied": True,
        "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
        "n_roi": 800, "n_total": 960,
    },
    {  # nbin-eligible (nchan=7680), single pointing
        "spw": "29", "nchan": 7680, "ratio_satisfied": False,
        "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
        "n_roi": 1000, "n_total": 7680,
    },
    {  # not nbin-eligible at all (ratio condition not met), tight PB already
        "spw": "31", "nchan": 1920, "ratio_satisfied": False,
        "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
        "n_roi": 1200, "n_total": 1920,
    },
]

# Predicted sizes per spw (GB). In the real pipeline these come from
# calculate_sizes() on the makeimlist targets.
cube_sizes = {"25": 15.0, "27": 15.0, "29": 70.0, "31": 15.0}
product_sizes = {"25": 50.0, "27": 50.0, "29": 150.0, "31": 50.0}

# Fields/targets are a property of the whole MOUS, not of a single spw:
# ONE field count for all spws. product_sizes above are per spw, summed
# over all fields.
nfields = 3

# 2. Build the per-lever mitigation table

In [3]:
table = build_mitigation_table(spws, use_roi=False, nfields=nfields)
table

,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction
spw,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,3,0.666667,0.933441
27,0.5,0.510000,0.265306,0.0,3,3,0.666667,0.940000
29,0.5,0.234375,0.437500,0.0,3,3,0.666667,0.928223
31,0.0,0.000000,0.577500,0.0,3,3,0.666667,0.859167


# 3. Set the size limits (GB)

In [4]:
limits = Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)
limits


Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)

# 4. Run the mitigation in one call. spw "29" is the representative spw here

In [5]:
final, msg = plan(
    table,
    limits,
    cube_sizes,
    product_sizes,
    repr_spw="29",
    nfields=nfields,
)

# 5. Display the outcome.

In [6]:
print(f"Result: {msg['shortmsg']}")
print("\nLever scores (product-weighted mitigation, best first):")
print(msg["lever_scores"].round(3).to_string())
print(f"\nSpws kept: {list(final.index[final['kept']])}")

Result: Size was mitigated

Lever scores (product-weighted mitigation, best first):
nfields    0.667
n_bin      0.417
cell       0.416
PB         0.268
ROI        0.000

Spws kept: ['25', '27', '29', '31']


In [7]:
final

,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction,cube_size_GB,product_size_GB,required_reduction,min_steps,levers_used,achieved_reduction,field_reduction,mitigated_cube_GB,mitigated_product_GB,kept
spw,,,,,,,,,,,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,1,0.666667,0.933441,15.0,50.0,0.333333,3,"n_bin,nfields,cell",0.669922,0.666667,4.951172,5.501302,True
27,0.5,0.510000,0.265306,0.0,3,1,0.666667,0.940000,15.0,50.0,0.333333,4,"PB,nfields,n_bin,cell",0.820000,0.666667,2.700000,3.000000,True
29,0.5,0.234375,0.437500,0.0,3,1,0.666667,0.928223,70.0,150.0,0.857143,4,"n_bin,cell,PB,nfields",0.784668,0.666667,15.073242,10.766602,True
31,0.0,0.000000,0.577500,0.0,3,1,0.666667,0.859167,15.0,50.0,0.333333,2,"cell,nfields",0.577500,0.666667,6.337500,7.041667,True
